In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

Cloning into 'kcw-analytics'...
remote: Enumerating objects: 587, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 587 (delta 106), reused 32 (delta 20), pack-reused 438 (from 1)
Receiving objects: 100% (587/587), 492.00 KiB | 5.93 MiB/s, done.
Resolving deltas: 100% (390/390), done.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [2]:

folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [3]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")



Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (2970, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (27770, 41)
Loaded: raw_syp_simas_sales_bills.csv -> (12876, 49)
Loaded: raw_syp_sidet_sales_lines.csv -> (37824, 38)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50274, 49)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154052, 41)
Loaded: raw_hq_sidet_sales_lines.csv -> (733417, 38)
Loaded: raw_hq_icmas_products.csv -> (114950, 94)
Loaded: raw_hq_armas_receivable.csv -> (2227, 20)
Loaded: raw_hq_apmas_payable.csv -> (978, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (275967, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)


In [4]:

import sys
import importlib

# ensure repo is on path
repo_path = f"{BASE_FOLDER_GIT}/kcw-analytics"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# import the module (NOT individual functions)
import src.kcw.utils as utils

# reload to pick up latest .py changes
importlib.reload(utils)



<module 'src.kcw.utils' from '/content/kcw-analytics/src/kcw/utils.py'>

In [123]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()
df_pimas = data["raw_hq_pimas_purchase_bills.csv"].copy()

In [124]:
df_simas.columns

Index(['ID', 'JOURMODE', 'JOURTYPE', 'JOURDATE', 'JOURNO', 'JOURTIME',
       'DEPTNO', 'BOOKNO', 'BILLTYPE', 'BILLDATE', 'BILLTIME', 'BILLNO',
       'LINES', 'TAXIC', 'DISCOUNT', 'DEDUCT', 'BEFORETAX', 'VAT', 'TAX',
       'AFTERTAX', 'EXEMPT', 'SVCCHG', 'WITHHOLD', 'PAID', 'CASHED', 'CASHAMT',
       'CHKAMT', 'DUEAMT', 'PAYSTAT', 'ACCTNO', 'ACCTNAME', 'ADDR1', 'ADDR2',
       'PO', 'SALE', 'RE', 'TERM', 'DUEDATE', 'NOTEDATE', 'NOTENO',
       'VOUCDATE1', 'VOUCNO1', 'VOUCDATE2', 'VOUCNO2', 'POSTED1', 'POSTED2',
       'REMARKS', 'CANCELED', 'DONE'],
      dtype='object')

In [179]:
cutoff = pd.Timestamp.today().replace(day=1)

YEAR = cutoff.year
MONTH = cutoff.month

cutoff

Timestamp('2026-03-01 22:10:08.003697')

In [180]:
#TO BE REMOVED
from datetime import datetime

YEAR = 2026
MONTH = 2

cutoff = datetime(YEAR, MONTH, 1)

In [181]:
def normalize_acctname(name):
    if pd.isna(name):
        return name

    n = str(name).lower()

    for k in ["lazada", "shopee", "tiktok"]:
        if k in n:
            return f"คุณลูกค้าทั่วไป {k}"

    return name


In [182]:
df = df_simas

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"].astype(str).str.strip(), errors="coerce")
df["VOUCDATE2"] = pd.to_datetime(df["VOUCDATE2"], errors="coerce")

mask_billno = df["BILLNO"].str.contains(r"TD|TR|TAD|CN", na=False)
mask_due_pos = df["DUEAMT"] != 0
mask_billbefore = df["BILLDATE"] < cutoff
mask_paidafter = df["VOUCDATE2"] > cutoff
mask_null = df["VOUCDATE2"].isna()
mask_cancel = df["CANCELED"].astype(str).str.strip().str.upper() == "N"
mask_vat = pd.to_numeric(df["VAT"], errors="coerce").fillna(0) > 0

df["ACCTNAME"] = df["ACCTNAME"].apply(normalize_acctname)

df_ar_sale = df[mask_billno & mask_due_pos & mask_billbefore & (mask_null | mask_paidafter) & mask_cancel & mask_vat]

df_ar_sale = df_ar_sale.sort_values(by=["ACCTNAME", "BILLDATE"])

In [183]:
df_ar_sale[["BILLNO", "BILLDATE","ACCTNO", "ACCTNAME", "VOUCDATE1", "VOUCDATE2", "PAID","DUEAMT", "VAT"]]

,BILLNO,BILLDATE,ACCTNO,ACCTNAME,VOUCDATE1,VOUCDATE2,PAID,DUEAMT,VAT
270884,TD6901-132,2026-01-31,7ธรพ,คุณ ธีรพงศ์ ฤทธิไกร,NaN,NaT,N,950.0,7.0
270882,TD6901-131,2026-01-31,7บธ,คุณ บุญธรรม สามารถ,NaN,NaT,N,3500.0,7.0
271002,TD6901-135,2026-01-31,7วงศ,คุณ วงษ์สวัสดิ์ จำปาทอง,NaN,NaT,N,1715.0,7.0
271814,TD6901-136,2026-01-31,7รวฒ,คุณ ศิริวุฒิ โสตถิรัตนพันธ์,NaN,NaT,N,6580.0,7.0
270888,TD6901-134,2026-01-31,7สทน,คุณ สุนทร อามาต,NaN,NaT,N,4450.0,7.0
...,...,...,...,...,...,...,...,...,...
270270,TD6901-129,2026-01-31,7ทมซ,บริษัท ไทยไม้ซุง จำกัด (สำนักงานใหญ่),NaN,NaT,N,4520.0,7.0
266426,TD6901-023,2026-01-09,7กช,หจก. ส.กิจชัยคอนกรีต (สาขาที่ 00002),NaN,2026-02-13,Y,3720.0,7.0
249175,TD6810-003,2025-10-01,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,2026-03-04,Y,18860.0,7.0
261324,TD6812-052,2025-12-10,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,2026-03-04,Y,25720.0,7.0


In [184]:
df = df_ar_sale

df_ar_summary = (
    df
    .groupby("ACCTNAME")
    .agg(
        bills=("BILLNO", "nunique"),
        total_amount=("DUEAMT", "sum")
    )
    .reset_index()
)

In [185]:
df_ar_summary

,ACCTNAME,bills,total_amount
0,คุณ ธีรพงศ์ ฤทธิไกร,1,950.00
1,คุณ บุญธรรม สามารถ,1,3500.00
2,คุณ วงษ์สวัสดิ์ จำปาทอง,1,1715.00
3,คุณ ศิริวุฒิ โสตถิรัตนพันธ์,1,6580.00
4,คุณ สุนทร อามาต,1,4450.00
5,คุณ อาบีดีน โด่หลี,1,1940.00
6,คุณ เดชา เกรียงไกร,1,125.00
7,คุณลูกค้าทั่วไป lazada,120,199158.25
8,คุณลูกค้าทั่วไป shopee,43,78900.00
9,คุณลูกค้าทั่วไป tiktok,32,23522.00


In [186]:
df = df_pimas

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"].astype(str).str.strip(), errors="coerce")
df["VOUCDATE2"] = pd.to_datetime(df["VOUCDATE2"], errors="coerce")

mask_due_pos = df["DUEAMT"] != 0
mask_billbefore = df["BILLDATE"] < cutoff
mask_paidafter = df["VOUCDATE2"] > cutoff
mask_null = df["VOUCDATE2"].isna()
mask_cancel = df["CANCELED"].astype(str).str.strip().str.upper() == "N"
mask_vat = pd.to_numeric(df["VAT"], errors="coerce").fillna(0) > 0

df["ACCTNAME"] = df["ACCTNAME"].apply(normalize_acctname)

df_ap_purchase = df[mask_due_pos & mask_billbefore & (mask_null | mask_paidafter) & mask_cancel & mask_vat]

df_ap_purchase = df_ap_purchase.sort_values(by=["ACCTNAME", "BILLDATE"])

In [187]:
df_ap_purchase[["BILLNO", "BILLDATE","ACCTNO", "ACCTNAME", "VOUCDATE1", "VOUCDATE2", "PAID","DUEAMT", "VAT"]]

,BILLNO,BILLDATE,ACCTNO,ACCTNAME,VOUCDATE1,VOUCDATE2,PAID,DUEAMT,VAT
47886,IV68121304,2025-12-13,7PRW,บจก. ประวิทย์ ออโตพาร์ท อิมปอร์ต-เอ็กซปอร์ต (ส...,NaN,2026-02-24,Y,1897.75,7.0
49197,IV69012611,2026-01-26,7PRW,บจก. ประวิทย์ ออโตพาร์ท อิมปอร์ต-เอ็กซปอร์ต (ส...,NaN,NaT,N,4010.36,7.0
47915,BV6812000012,2025-12-13,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,2026-02-24,Y,4276.04,7.0
48011,BV6812000014,2025-12-16,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,2026-02-24,Y,3950.60,7.0
48466,BV6901000002,2026-01-06,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,8988.75,7.0
...,...,...,...,...,...,...,...,...,...
48902,IV26010234,2026-01-17,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,4824.51,7.0
48985,IV26010292,2026-01-20,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,7457.90,7.0
49000,IV26010301,2026-01-21,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,3356.39,7.0
49002,IV26010319,2026-01-21,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,2085.38,7.0


In [188]:
df = df_ap_purchase

df_ap_summary = (
    df
    .groupby("ACCTNAME")
    .agg(
        bills=("BILLNO", "nunique"),
        total_amount=("DUEAMT", "sum")
    )
    .reset_index()
)

In [189]:
df_ap_summary

,ACCTNAME,bills,total_amount
0,บจก. ประวิทย์ ออโตพาร์ท อิมปอร์ต-เอ็กซปอร์ต (ส...,2,5908.11
1,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,8,35168.17
2,บจก. เจเจวาย ออโต้ เวิร์คส์ (สำนักงานใหญ่),2,3124.40
3,บจก. เอส ที เค กรุ๊ป 2008 (สำนักงานใหญ่),19,68066.67
4,บจก. เอส.พี.อาร์.วาย ออโต้พาร์ท (สำนักงานใหญ่),27,27429.45
...,...,...,...
88,ห้างหุ้นส่วนจำกัด พี.อี.ออโต้เทรด (สำนักงานใหญ่),5,14990.01
89,ห้างหุ้นส่วนจำกัด รุ่งเรืองทรัพย์ทวี กลการ (สนญ.),1,25252.00
90,ห้างหุ้นส่วนจำกัด ฮ.อาหลั่ยยนต์ (สำนักงานใหญ่),5,23358.10
91,ห้างหุ้นส่วนจำกัด เอส.เค.ที. แมชชีนทูลส์ (สนญ.),38,76230.36


In [190]:
def add_total_row(df, label_col="ACCTNAME", amount_col="total_amount"):
    total_row = pd.DataFrame([{
        label_col: "TOTAL",
        "bills": df["bills"].sum(),
        amount_col: df[amount_col].sum()
    }])

    return pd.concat([df, total_row], ignore_index=True)

In [191]:
def rename_and_keep(df, rename_map):
    cols = [c for c in rename_map.keys() if c in df.columns]
    return df[cols].rename(columns=rename_map)

In [192]:
df_ap_summary = add_total_row(df_ap_summary)
df_ar_summary = add_total_row(df_ar_summary)

In [193]:
rename_map = {
    "ACCTNAME": "ชื่อลูกหนี้",
    "BILLNO": "เลขที่บิล",
    "BILLDATE": "วันที่",
    "DUEAMT": "จำนวน",
    "bills": "จำนวนบิล",
    "total_amount": "ยอดรวม",
}

df_ar_sale     = rename_and_keep(df_ar_sale, rename_map)
df_ar_summary  = rename_and_keep(df_ar_summary, rename_map)

rename_map["ACCTNAME"] = "ชื่อเจ้าหนี้"

df_ap_purchase      = rename_and_keep(df_ap_purchase , rename_map)
df_ap_summary    = rename_and_keep(df_ap_summary  , rename_map)

In [194]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "AR_AP_Report",
    f"ar_ap_report_{YEAR}_{MONTH:02}.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [196]:
!pip install XlsxWriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.3 MB/s eta 0:00:00


In [197]:
import pandas as pd

output_file = folder

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:

    df_ap_purchase.to_excel(writer, sheet_name="AP Purchase", index=False)
    df_ap_summary.to_excel(writer, sheet_name="AP Summary", index=False)
    df_ar_sale.to_excel(writer, sheet_name="AR Sale", index=False)
    df_ar_summary.to_excel(writer, sheet_name="AR Summary", index=False)

    workbook = writer.book

    header_format = workbook.add_format({
        "bold": True,
        "align": "center",
        "valign": "vcenter",
        "border": 1
    })

    money_format = workbook.add_format({
        "num_format": "#,##0.00"
    })

    for sheet_name, df in {
        "AP Purchase": df_ap_purchase,
        "AP Summary": df_ap_summary,
        "AR Sale": df_ar_sale,
        "AR Summary": df_ar_summary
    }.items():

        worksheet = writer.sheets[sheet_name]

        # freeze header
        worksheet.freeze_panes(1, 0)

        # rewrite header with style
        for col_num, col_name in enumerate(df.columns):
            worksheet.write(0, col_num, col_name, header_format)

        # auto column width
        for i, col in enumerate(df.columns):
            width = max(df[col].astype(str).map(len).max(), len(col)) + 2
            worksheet.set_column(i, i, width)

        # apply currency format to amount columns
        for i, col in enumerate(df.columns):
            if col in ["จำนวน", "ยอดรวม"]:
                worksheet.set_column(i, i, None, money_format)